# 实验结果汇总
使用 Notebook 聚合 `cwPSU/results` 目录中的 CSV，计算相同 `bitlength`、`mu`、`hw` 的平均指标并生成汇总表。

In [16]:
# Section 1 – Setup Paths and Imports
import re
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
CANDIDATE_DIRS = [
    NOTEBOOK_DIR.parent / "results",
    NOTEBOOK_DIR.parent.parent / "results",
    NOTEBOOK_DIR / "results",
    NOTEBOOK_DIR / ".." / "results",
    NOTEBOOK_DIR / "../results"
]
RESULTS_DIR = None
fallback_dir = None
for candidate in CANDIDATE_DIRS:
    try:
        resolved = candidate.resolve()
    except FileNotFoundError:
        continue
    if not resolved.is_dir():
        continue
    if fallback_dir is None:
        fallback_dir = resolved
    has_bitlength_files = any(resolved.glob("bitlength_*hw_*_*.csv"))
    if has_bitlength_files:
        RESULTS_DIR = resolved
        break
if RESULTS_DIR is None:
    if fallback_dir is not None:
        RESULTS_DIR = fallback_dir
    else:
        raise FileNotFoundError("未能定位 results 目录，请确认目录结构")

OUTPUT_PATH = NOTEBOOK_DIR / "results_summary.csv"
MB = 1024 ** 2

print(f"读取结果目录: {RESULTS_DIR}")
print(f"输出汇总文件: {OUTPUT_PATH}")

读取结果目录: /home/jrj/code/cwPSU/results
输出汇总文件: /home/jrj/code/cwPSU/experiments/results_summary.csv


In [17]:
# Section 2 – Load CSV Results
# 解析文件名获取 bitlength/hw/运行编号/网络环境 信息
prefix_pattern = re.compile(r"^bitlength_(\\d+)hw_(\\d+)")
records = []
unmatched_files = []

for csv_path in sorted(RESULTS_DIR.glob("*.csv")):
    stem = csv_path.stem
    if not stem.startswith("bitlength_"):
        continue
    try:
        prefix, network_env_raw = stem.rsplit("_", 1)
    except ValueError:
        unmatched_files.append(csv_path.name)
        continue

    network_env = network_env_raw.lower()
    prefix = prefix.rstrip("_")

    prefix_match = prefix_pattern.match(prefix)
    if not prefix_match:
        unmatched_files.append(csv_path.name)
        continue

    bitlength_from_name = int(prefix_match.group(1))
    hw_from_name = int(prefix_match.group(2))

    remainder = prefix[prefix_match.end():]
    remainder_parts = [part for part in remainder.split("_") if part]

    run_index = None
    if remainder_parts:
        try:
            run_index = int(remainder_parts[-1])
        except ValueError:
            unmatched_files.append(csv_path.name)
            continue

    try:
        raw = pd.read_csv(csv_path, header=None, names=["metric", "value"])
    except pd.errors.EmptyDataError:
        continue
    except Exception as exc:
        unmatched_files.append(f"{csv_path.name} (read error: {exc})")
        continue

    if raw.empty:
        continue

    record = raw.set_index("metric")["value"].to_dict()

    record.update({
        "source_file": csv_path.name,
        "network_env": network_env,
        "run_index": run_index,
    })

    if "bitlength" not in record or pd.isna(record["bitlength"]):
        record["bitlength"] = bitlength_from_name
    if "hw" not in record or pd.isna(record["hw"]):
        record["hw"] = hw_from_name

    records.append(record)

if not records:
    unmatched_msg = ""
    if unmatched_files:
        unmatched_preview = ", ".join(unmatched_files[:8])
        unmatched_msg = f"。可能的未匹配文件示例: {unmatched_preview}"
    raise ValueError(f"在 {RESULTS_DIR} 中未发现符合命名规则的结果 CSV{unmatched_msg}")

metrics_df = pd.DataFrame(records)

if unmatched_files:
    print("以下文件未按预期格式解析:")
    for name in unmatched_files:
        print(f"  - {name}")

metrics_df["network_env"] = metrics_df["network_env"].astype("string")
metrics_df["source_file"] = metrics_df["source_file"].astype("string")

for column in ["bitlength", "hw", "mu", "run_index"]:
    if column in metrics_df.columns:
        metrics_df[column] = pd.to_numeric(metrics_df[column], errors="coerce")

comm_columns = [
    "comm_request", "comm_response", "comm_final_response",
    "comm_gal_keys", "comm_params",
    "comm_permute_share_request_bytes", "comm_permute_share_response_bytes",
    "comm_pk", "comm_relin_keys",
]
for col in comm_columns:
    if col not in metrics_df.columns:
        metrics_df[col] = 0.0
    metrics_df[col] = metrics_df[col].fillna(0.0) / MB

time_columns = [
    "time_extract_response", "time_server_latency",
    "time_permute_share_client_compute_us", "time_permute_share_network_ms",
    "time_setup_crypto"
]
for col in time_columns:
    if col not in metrics_df.columns:
        metrics_df[col] = 0.0
    metrics_df[col] = metrics_df[col].fillna(0.0)

metrics_df["time_permute_share_client_compute_ms"] = (
    metrics_df["time_permute_share_client_compute_us"] / 1000.0
)

metrics_df["online_comm"] = (
    metrics_df[["comm_request", "comm_response", "comm_final_response"]].sum(axis=1)
)
metrics_df["offline_comm"] = (
    metrics_df[[
        "comm_gal_keys", "comm_params",
        "comm_permute_share_request_bytes", "comm_permute_share_response_bytes",
        "comm_pk", "comm_relin_keys",
    ]].sum(axis=1)
)
metrics_df["online_runtime"] = (
    metrics_df[["time_extract_response", "time_server_latency"]].sum(axis=1)
)
metrics_df["offline_runtime"] = (
    metrics_df["time_permute_share_client_compute_ms"]
    + metrics_df["time_permute_share_network_ms"]
    + metrics_df["time_setup_crypto"]
)

sort_cols = ["bitlength", "hw", "mu", "network_env"]
if "run_index" in metrics_df.columns:
    sort_cols.append("run_index")

base_cols = ["source_file", "bitlength", "hw", "mu", "network_env"]
if "run_index" in metrics_df.columns:
    base_cols.append("run_index")
other_cols = [col for col in metrics_df.columns if col not in base_cols]

metrics_df = (
    metrics_df
    .sort_values(sort_cols, na_position="last")
    .reset_index(drop=True)
)

ordered_cols = base_cols + other_cols

print(f"读取到 {len(metrics_df)} 份实验记录")
metrics_df[ordered_cols]

ValueError: 在 /home/jrj/code/cwPSU/results 中未发现符合命名规则的结果 CSV。可能的未匹配文件示例: bitlength_128hw_19_10__wan-1.csv, bitlength_128hw_19_11__wan-1.csv, bitlength_128hw_19_12__wan-1.csv, bitlength_128hw_19_1__lan.csv, bitlength_128hw_19_2__lan.csv, bitlength_128hw_19_3__lan.csv, bitlength_128hw_19_4__wan-100.csv, bitlength_128hw_19_5__wan-100.csv

In [ ]:
# Section 3 – Aggregate by Parameters
group_cols = ["bitlength", "hw", "mu", "network_env"]
missing = [col for col in group_cols if col not in metrics_df.columns]
if missing:
    raise KeyError(f"缺少必要列: {missing}")

numeric_cols = metrics_df.select_dtypes(include="number").columns.tolist()
value_cols = [col for col in numeric_cols if col not in set(group_cols + ["run_index"])]

aggregated_df = (
    metrics_df
    .groupby(group_cols, dropna=False)[value_cols]
    .mean()
    .reset_index()
    .sort_values(group_cols)
    .reset_index(drop=True)
)

run_counts = (
    metrics_df.groupby(group_cols, dropna=False).size().rename("run_count").reset_index()
)
aggregated_df = aggregated_df.merge(run_counts, on=group_cols, how="left")

print(f"聚合后得到 {len(aggregated_df)} 组参数")
aggregated_df.head()

聚合后得到 2 组参数


,bitlength,mu,hw,comm_final_response,comm_gal_keys,comm_params,comm_pk,comm_ps,comm_relin_keys,comm_request,...,comp_server_ps,effective_lambda,ell,log_poly_modulus_degree,task,time_extract_response,time_server,time_server_extract,time_server_latency,time_setup_crypto
0,32,526,7,3.867543e+05,4.760150e+07,85.0,251209.818182,5050456.0,9.930272e+05,5.076949e+06,...,127.636364,19.0,26.0,13.0,0.0,1.644401e+05,7005.909091,2.0,7036.818182,32.909091
1,128,296,19,1.123897e+06,2.103463e+08,92.0,674260.818182,10883160.0,4.045754e+06,2.906555e+08,...,162.363636,114.0,517.0,14.0,0.0,1.408281e+06,55389.727273,7.0,57407.272727,88.090909


In [ ]:
# Section 4 – Prepare Summary Table
summary_df = aggregated_df.copy()

display_cols = [
    "bitlength", "hw", "mu", "network_env", "run_count",
    "online_comm", "offline_comm",
    "comm_request", "comm_response", "comm_final_response",
    "comm_gal_keys", "comm_params",
    "comm_permute_share_request_bytes", "comm_permute_share_response_bytes",
    "comm_pk", "comm_relin_keys",
    "online_runtime", "offline_runtime",
    "time_extract_response", "time_server_latency",
    "time_permute_share_network_ms", "time_permute_share_client_compute_ms",
    "time_setup_crypto"
]

for col in display_cols:
    if col not in summary_df.columns:
        summary_df[col] = float("nan")

summary_df = summary_df[display_cols]
summary_df.head()

,bitlength,mu,hw,comm_final_response,comm_gal_keys,comm_params,comm_pk,comm_ps,comm_relin_keys,comm_request,...,time_extract_response,time_server,time_server_extract,time_server_latency,time_setup_crypto,keys,request,response,ps,final_response
0,32,526,7,3.867543e+05,4.760150e+07,85.0,251209.818182,5050456.0,9.930272e+05,5.076949e+06,...,1.644401e+05,7005.909091,2.0,7036.818182,32.909091,46.343353,4.841756,0.737888,4.81649,0.368838
1,128,296,19,1.123897e+06,2.103463e+08,92.0,674260.818182,10883160.0,4.045754e+06,2.906555e+08,...,1.408281e+06,55389.727273,7.0,57407.272727,88.090909,204.460189,277.190652,2.143736,10.37899,1.071831


In [ ]:
# Section 5 – Display Aggregated Metrics
report_df = (
    summary_df
    .sort_values(["bitlength", "hw", "mu", "network_env"], na_position="last")
    .reset_index(drop=True)
)
report_df

,bitlength,mu,hw,keys,request,response,ps,final_response,time_setup_crypto,time_server,time_server_extract,time_extract_response
0,32,526,7,46.343353,4.841756,0.737888,4.81649,0.368838,32.909091,7005.909091,2.0,1.644401e+05
1,128,296,19,204.460189,277.190652,2.143736,10.37899,1.071831,88.090909,55389.727273,7.0,1.408281e+06


In [ ]:
# Section 6 – Persist Processed Data
report_df.to_csv(OUTPUT_PATH, index=False)
print(f"已保存汇总结果至 {OUTPUT_PATH}")